In [1]:
import pandas as pd
import time
import os
import re

In [2]:
dir="../data/"
for f in os.listdir(dir):
    if re.match("question_answer.*.csv",f):
        print(f)

question_answer.1763481918.5790632.csv
question_answer.1763480211.4647102.csv
question_answer.1763476709.7714555.csv
question_answer.1763478542.9023702.csv


In [3]:
filenames=[dir+f for f in os.listdir(dir) if re.match("question_answer.*.csv",f)]

In [4]:
filenames

['../data/question_answer.1763481918.5790632.csv',
 '../data/question_answer.1763480211.4647102.csv',
 '../data/question_answer.1763476709.7714555.csv',
 '../data/question_answer.1763478542.9023702.csv']

In [5]:
columns=["idx","text","text_id","question","answer"]
#df=pd.read_csv(filen)

In [6]:
dfs=[]
for file in filenames:
    df=pd.read_csv(file,index_col=None,header=0)
    dfs.append(df)

df=pd.concat(dfs,axis=0,ignore_index=True)

In [7]:
x=df.text_id.unique().tolist()

In [8]:
x2=[35, 351, 331, 352, 123,  84, 186, 237, 364, 238, 358, 268, 100,
        31, 174, 315, 134, 274, 153,  69,  57, 291,  48,  52,  93, 159,
         9, 298,  11]

In [9]:
df.count()

idx         279
text        279
text_id     279
question    278
answer      278
dtype: int64

In [10]:
df=df.drop(df[df.question.isna()].index,axis=0)

In [11]:
df.count()

idx         278
text        278
text_id     278
question    278
answer      278
dtype: int64

In [12]:
df=df.drop(df[df.answer.isna()].index,axis=0)

In [13]:
df.count()

idx         278
text        278
text_id     278
question    278
answer      278
dtype: int64

In [14]:
df=df.drop(df[df.question.str.contains('passage')].index,axis=0)
df=df.drop(df[df.question.str.contains('text')].index,axis=0)
df=df.drop(df[df.question.str.contains('sentence')].index,axis=0)

In [15]:
df.count()

idx         214
text        214
text_id     214
question    214
answer      214
dtype: int64

In [16]:
df.head()

,idx,text,text_id,question,answer
0,0,"Of course this was not the original assertion,...",27,What does Thrasymachus originally assert about...,Thrasymachus originally asserts that the ruler...
2,2,"Of course this was not the original assertion,...",27,What is Socrates' attitude towards changing wo...,Socrates is not disposed to quarrel about word...
3,3,"Of course this was not the original assertion,...",27,"According to the analogy of the arts, what doe...","Every art or science has an interest, which is..."
4,4,"Of course this was not the original assertion,...",27,What is justice' interest according to Socrate...,Justice has an interest which is the interest ...
11,11,"Still, mathematics admit of other applications...",175,What is criticized about both astronomers and ...,Both astronomers and Pythagorean harmonists ar...


In [17]:
import numpy as np
norm = np.linalg.norm
def cosine_distance(v1,v2):
    v1=np.array(v1)
    v2=np.array(v2)
    return v1.dot(v2)/norm(v1)/norm(v2)

In [18]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
df["text_vec"]=df.apply(lambda row: model.encode(row["text"]),axis=1)
df["question_vec"]=df.apply(lambda row: model.encode(row["question"]),axis=1)

In [20]:
df["question_distance"]=df.apply(lambda row: cosine_distance(row["text_vec"],row["question_vec"]),axis=1)

In [21]:
df=df[df["question_distance"]>.7]

In [22]:
df.count()

idx                  17
text                 17
text_id              17
question             17
answer               17
text_vec             17
question_vec         17
question_distance    17
dtype: int64

In [23]:
df=df.drop(["idx","text_vec","question_vec","question_distance"],axis=1)

In [38]:
df.to_csv(f"../data/quality_q_a.{time.time()}.csv")

In [36]:
df=df.drop(["index"],axis=1)

In [37]:
df

,text,text_id,question,answer
0,"Of course this was not the original assertion,...",27,What does Thrasymachus originally assert about...,Thrasymachus originally asserts that the ruler...
1,"Of course this was not the original assertion,...",27,What is Socrates' attitude towards changing wo...,Socrates is not disposed to quarrel about word...
2,"Still, mathematics admit of other applications...",175,What is criticized about both astronomers and ...,Both astronomers and Pythagorean harmonists ar...
3,"The ‘New Atlantis’ is only a fragment, and far...",369,How does Lord Bacon's portrayal of the governo...,Lord Bacon minutely describes the governor's d...
4,"On the other hand, Plato is far in advance of ...",341,What does Plato teach about education?,Plato teaches that education is to be continue...
5,"On the other hand, Plato is far in advance of ...",341,Would Plato allow any kind of education to cease?,"No, Plato would never allow education of some ..."
6,"(1) The Republic, though probably written at i...",349,During which phase of Plato's life were the La...,The Laws were certainly composed during Plato'...
7,That Plato should have emancipated himself fro...,297,What was the typical role of women in ancient ...,They were not entertainers or mistresses of th...
8,That Plato should have emancipated himself fro...,297,How did Greek society view the ideal of womanh...,The historian's conception of feminine excelle...
9,What Plato had heard or seen of Sparta was app...,313,What was Plato's mistaken application of what ...,Plato applied his observations of Spartan supe...


In [39]:
from qdrant_client import QdrantClient

def get_qdrant_client():
    """Create a singleton Qdrant client."""
    return QdrantClient("http://localhost:6333")

In [40]:
client = get_qdrant_client()

In [47]:
from typing import Dict, List
collection_name='simple_rag'

def semantic_search(
    query: str, collection_name: str, top_k: int = 5
) -> List[Dict]:
    """Perform semantic search on code chunks."""
    contexts=[]
    qry_vec = model.encode(query).tolist() 
    client = get_qdrant_client()

    try:
        results = client.query_points(
            collection_name=collection_name, query=qry_vec, limit=top_k
        )
        return [
            {
                "id": hit.id,
                "file_path": hit.payload["file_path"],
                "book": hit.payload["book"],
                "text": hit.payload["text"],
                "score": hit.score,
            }
            for hit in results.points
        ]
    except Exception as e:
        print(f"Search error: {e}")
        return []

In [45]:
queries=df[["question"]].to_dict()
queries=queries['question']

In [67]:
texts=df[["text_id","text"]].drop_duplicates()

In [84]:
import pickle

with open("../data/corpus_all.pkl","rb") as f:
    corpus=pickle.load(f)

In [85]:
results=[]
for k in queries.keys():
    result=semantic_search(queries[k],collection_name=collection_name)
    text_ids=[i["id"] for i in result]
    #print(text_ids)
    results.append([k,text_ids])

In [88]:
similarities_results=[]
for r in results:
    q_vec=model.encode(queries[r[0]]).tolist()
    #print(r[0],q_vec)
    similarities=[]
    for t in r[1]:
        t_vec=model.encode(corpus[t]).tolist()
        sim=cosine_distance(q_vec,t_vec)
        if sim > 0.7:
            similarities.append([t,sim])
    similarities_results.append([r[0],similarities])

In [24]:
filenames=["quality_q_a.1763447787.7302473.csv","quality_q_a.1763449815.1628058.csv","quality_q_a.1763454216.9625602.csv","quality_q_a.csv"]

In [26]:
dfs=[]
for file in filenames:
    df=pd.read_csv(dir+file,index_col=None,header=0)
    dfs.append(df)

df=pd.concat(dfs,axis=0,ignore_index=True)

In [28]:
df=df.drop(["idx","text_vec","question_vec","question_distance","id","Unnamed: 0"],axis=1)

In [35]:
df.sort_values(by=["text_id","question"]).drop_duplicates().reset_index(drop=True)

,text,text_id,question,answer
0,In what may be called the epilogue of the disc...,41,What does Plato argue about the nature of evil?,Plato argues that evil is not a principle of s...
1,"In the third book of the Republic, Plato prese...",86,What does Plato say about true art according t...,True art is not fanciful and imitative but sim...
2,There is a difficulty in understanding what Pl...,122,What does Plato imply by 'the longer way'?,Plato seems to intimate some metaphysic of the...
3,Here Adeimantus interposes:—‘No man can answer...,144,How does Socrates respond to Adeimantus' argum...,Socrates agrees with Adeimantus that it is qui...
4,There is more difficulty in comprehending how ...,162,Why does Plato use four terms instead of three...,Probably Plato has been led by the love of ana...
5,The modern mathematician will readily sympathi...,182,What is noted about the state of solid geometr...,The text mentions that solid geometry was not ...
6,The distinction between the mathematician and ...,190,What is the main distinction between the mathe...,The faculty of the mathematician is quite dist...
7,In the previous books Plato has described the ...,205,How do Plato describe these perverted or decli...,He describes them in a succession of parallels...
8,Plato begins by speaking of a perfect or cycli...,219,How is a perfect number defined according to P...,A number in which the sum of the divisors equa...
9,Plato begins by speaking of a perfect or cycli...,219,How is a perfect number defined according to P...,A number in which the sum of the divisors equa...


In [36]:
df.to_csv("../data/cleanse_q_a.csv")

In [18]:
df0=pd.read_csv(files[0])
df1=pd.read_csv(files[1])

In [19]:
print("df0: ",df0.columns)
print("df1: ",df1.columns)

df0:  Index(['Unnamed: 0', 'idx', 'text', 'text_id', 'question', 'answer'], dtype='object')
df1:  Index(['Unnamed: 0', 'idx', 'text', 'text_id', 'question', 'answer'], dtype='object')


In [22]:
df_results=pd.concat([df0,df1])

In [23]:
df_results.count()

Unnamed: 0    73
idx           73
text          73
text_id       73
question      73
answer        73
dtype: int64

In [25]:
df_results.to_csv('../data/question_answer.2.csv')

In [26]:
df=pd.read_csv('../data/question_answer.2.csv',index_col="idx")

In [27]:
df

,Unnamed: 0.1,Unnamed: 0,text,text_id,question,answer
idx,,,,,,
16,0,16,The Idea of good is so called only in the Repu...,338,In which dialogue does the term 'Idea of good'...,The term 'Idea of good' appears only in the Re...
19,1,19,The Idea of good is so called only in the Repu...,338,How does Plato view the investigation of nature?,Plato views the investigation of nature as ano...
20,2,20,There remains to be considered the great diffi...,217,What is considered a great difficulty in this ...,The so-called number of the State
22,3,22,There remains to be considered the great diffi...,217,"According to Cicero, what is the number of the...",A proverb of obscurity (Ep. ad Att.)
27,4,27,The fifth book is the new beginning of the Rep...,138,What is the main theme introduced in the fifth...,The community of property and of family are fi...
...,...,...,...,...,...,...
72,45,72,Adeimantus objects first of all to the form of...,151,How can variations in meaning or assumptions o...,"In a long argument, words are apt to change th..."
73,46,73,"Lastly, no one can have observed the first ris...",317,What does the author feel about regulating pas...,The author feels that there is something unsat...
74,47,74,"Lastly, no one can have observed the first ris...",317,How should the most important influence on hum...,"According to the philosopher, the most importa..."


In [28]:
df=df[['text', 'text_id', 'question', 'answer']]

In [30]:
df.to_csv('../data/question_answer.2.csv')